In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# OpenQASM 3 y el SDK de Qiskit

<details>
<summary><b>Versiones de paquetes</b></summary>

El código de esta página fue desarrollado con los siguientes requisitos.
Recomendamos usar estas versiones o versiones más recientes.

```
qiskit[all]~=2.3.0
```
</details>

El SDK de Qiskit proporciona algunas herramientas para convertir entre representaciones OpenQASM de programas cuánticos y la clase QuantumCircuit. Ten en cuenta que estas herramientas aún se encuentran en una fase exploratoria de desarrollo y seguirán evolucionando a medida que el soporte de Qiskit para las capacidades de circuitos dinámicos expresadas por OpenQASM 3 vaya aumentando.

> **Note:** Esta función aún está en fase exploratoria. Por ello, es probable que la sintaxis y las capacidades evolucionen.

## Importar un programa OpenQASM 3 en Qiskit
Para usar esta función, debes instalar el paquete `qiskit_qasm3_import`. Instálalo con el siguiente comando.

In [1]:
import qiskit.qasm3

program = """
    OPENQASM 3.0;
    include "stdgates.inc";

    input float[64] a;
    qubit[3] q;
    bit[2] mid;
    bit[3] out;

    let aliased = q[0:1];

    gate my_gate(a) c, t {
      gphase(a / 2);
      ry(a) c;
      cx c, t;
    }
    gate my_phase(a) c {
      ctrl @ inv @ gphase(a) c;
    }

    my_gate(a * 2) aliased[0], q[{1, 2}][0];
    measure q[0] -> mid[0];
    measure q[1] -> mid[1];

    while (mid == "00") {
      reset q[0];
      reset q[1];
      my_gate(a) q[0], q[1];
      my_phase(a - pi/2) q[1];
      mid[0] = measure q[0];
      mid[1] = measure q[1];
    }

    if (mid[0]) {
      let inner_alias = q[{0, 1}];
      reset inner_alias;
    }

    out = measure q;
"""
circuit = qiskit.qasm3.loads(program)
circuit.draw("mpl")

<Image src="../docs/images/guides/interoperate-qiskit-qasm3/extracted-outputs/e805197c-51fb-4216-8d24-ae26633d29bc-0.svg" alt="Output of the previous code cell" />

Actualmente hay dos funciones de alto nivel disponibles para importar desde OpenQASM 3 a Qiskit. Estas funciones son `load()`, que recibe un nombre de archivo, y `loads()`, que recibe el programa directamente como cadena de texto:

In [2]:
from qiskit import QuantumCircuit
from qiskit.qasm3 import dumps

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

dumps(qc)

'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[2] meas;\nqubit[2] q;\nh q[0];\ncx q[0], q[1];\nbarrier q[0], q[1];\nmeas[0] = measure q[0];\nmeas[1] = measure q[1];\n'

En este ejemplo, definimos un programa cuántico usando OpenQASM 3 y usamos `loads()` para convertirlo directamente en un QuantumCircuit:

In [3]:
from qiskit import QuantumCircuit
from qiskit.qasm3 import dump

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

f = open("my_file.txt", "w")
dump(qc, f)
f.close()

![Salida de la celda de código anterior](../docs/images/guides/interoperate-qiskit-qasm3/extracted-outputs/e805197c-51fb-4216-8d24-ae26633d29bc-0.svg)

## Exportar a OpenQASM 3
Puedes exportar código de Qiskit a OpenQASM 3 con `dumps()`, que exporta a una cadena de texto, o `dump()`, que exporta a un archivo.

### Ejemplo con `dumps()`